In [ ]:
# ── Imports, AWS clients, and pipeline configuration ─────────────────────────

import json
import boto3
import base64
import re
import time
import math
import logging
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from transformers import AutoTokenizer

from feature_utils import (
    extract_url_features,
    FEATURE_COLS_ORDER,
    SUSPICIOUS_KEYWORDS,
    URL_SHORTENERS,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

URL_ENDPOINT_NAME = '<YOUR_URL_ENDPOINT_NAME>'

DOM_ENDPOINT_NAME = '<YOUR_DOM_ENDPOINT_NAME>'

VLM_MODEL_ID = 'us.anthropic.claude-sonnet-4-6'

AWS_REGION = 'us-east-1'

HIGH_CONFIDENCE_THRESHOLD = 0.85   # above this → Auto-Block
LOW_CONFIDENCE_THRESHOLD  = 0.25   # below this → Allow

VLM_TRIGGER_THRESHOLD = 0.40  # trigger VLM if combined score > this in medium band

MAX_SECTION_CHARS = 400
MAX_BODY_CHARS    = 1000
DOM_MAX_LEN       = 512

sm_runtime = boto3.client('sagemaker-runtime', region_name=AWS_REGION)  # Amazon SageMaker AI runtime
bedrock_rt = boto3.client('bedrock-runtime',   region_name=AWS_REGION)  # Amazon Bedrock runtime
tokenizer  = AutoTokenizer.from_pretrained('distilbert-base-uncased')

CLASS_ORDER = ['benign', 'phishing', 'defacement', 'malware']


class Decision(str, Enum):
    AUTO_BLOCK   = 'AUTO_BLOCK'
    HUMAN_REVIEW = 'HUMAN_REVIEW'
    ALLOW        = 'ALLOW'


@dataclass
class PipelineResult:
    url:           str
    decision:      Decision
    combined_score: float                   # 0.0 (benign) → 1.0 (phishing)
    url_score:     Optional[float] = None   # LightGBM P(phishing)
    dom_score:     Optional[float] = None   # DistilBERT P(phishing)
    vlm_score:     Optional[float] = None   # VLM P(phishing), None if not triggered
    vlm_triggered: bool = False
    vlm_reasoning: Optional[str]  = None
    url_model_raw: dict = field(default_factory=dict)
    dom_model_raw: dict = field(default_factory=dict)
    errors:        list = field(default_factory=list)
    latency_ms:    dict = field(default_factory=dict)


In [ ]:
# ── Stage 1a: LightGBM URL classifier ───────────────────────────────────────

def run_url_model(url: str) -> tuple[Optional[float], dict]:
    """
    Invoke the LightGBM URL classification endpoint.
    Expects text/csv input; returns {"probabilities": [[p_benign, p_phishing, p_defacement, p_malware]]}
    """
    try:
        t0 = time.time()

        feats = extract_url_features(url)
        row = pd.DataFrame(
            [[feats[c] for c in FEATURE_COLS_ORDER]],
            columns=FEATURE_COLS_ORDER,
        )
        csv_body = row.to_csv(header=False, index=False).encode('utf-8')

        resp = sm_runtime.invoke_endpoint(
            EndpointName=URL_ENDPOINT_NAME,
            ContentType='text/csv',
            Accept='application/json',
            Body=csv_body,
        )
        raw = json.loads(resp['Body'].read().decode('utf-8'))
        latency = round((time.time() - t0) * 1000, 1)
        logger.info(f'[URL Model] raw={raw} latency={latency}ms')

        probs = raw['probabilities'][0]

        p_benign    = float(probs[0])
        p_malicious = 1.0 - p_benign

        return round(p_malicious, 4), {
            'p_malicious':   round(p_malicious, 4),
            'multiclass':    CLASS_ORDER[int(np.argmax(probs))],
            'probabilities': {cls: round(float(p), 4) for cls, p in zip(CLASS_ORDER, probs)},
        }

    except Exception as e:
        logger.error(f'[URL Model] Error: {e}')
        return None, {'error': str(e)}


In [ ]:
# ── Stage 1b: DistilBERT DOM classifier ─────────────────────────────────────

def extract_dom_text(html: str, url: str = '') -> str:
    """
    Extract structured text from raw HTML for transformer classification.
    Sections ordered by phishing-signal importance (TEXT last for truncation).
    """
    try:
        soup = BeautifulSoup(html, 'lxml')
    except Exception:
        soup = BeautifulSoup(html, 'html.parser')

    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()

    parts = []

    title_tag = soup.find('title')
    title_txt = title_tag.get_text(strip=True)[:MAX_SECTION_CHARS] if title_tag else ''
    parts.append(f'[TITLE] {title_txt}')

    meta_vals = []
    for m in soup.find_all('meta', attrs={'name': True, 'content': True}):
        if m.get('name', '').lower() in ('description', 'keywords', 'author'):
            meta_vals.append(m['content'][:100])
    parts.append('[META] ' + ' | '.join(meta_vals))

    form_strs = []
    for form in soup.find_all('form'):
        action = form.get('action', '')
        method = form.get('method', 'get').upper()
        form_strs.append(f'{method}:{action[:80]}')
    parts.append('[FORMS] ' + ' | '.join(form_strs[:10]))

    input_types = [inp.get('type', 'text').lower() for inp in soup.find_all('input')]
    parts.append('[INPUTS] ' + ' '.join(input_types[:20]))

    page_host = url.split('/')[2] if url and '//' in url else ''
    ext_srcs  = []
    for tag in soup.find_all(['script', 'iframe', 'img', 'link']):
        src = tag.get('src') or tag.get('href') or ''
        if src.startswith('http') and page_host and page_host not in src:
            ext_srcs.append(src[:80])
    parts.append('[EXT_RESOURCES] ' + ' | '.join(ext_srcs[:10]))

    links = [a['href'][:80] for a in soup.find_all('a', href=True)]
    parts.append('[LINKS] ' + ' | '.join(links[:20]))

    body = soup.find('body')
    body_txt = body.get_text(separator=' ', strip=True)[:MAX_BODY_CHARS] if body else ''
    parts.append(f'[TEXT] {body_txt}')

    return ' '.join(parts)

In [ ]:

def run_dom_model(html: str, url: str = '') -> tuple[Optional[float], dict]:
    """
    Extract DOM features and invoke the DistilBERT endpoint.
    Returns (prob_phishing, raw_response_dict).
    """
    try:
        dom_text = extract_dom_text(html, url)

        tokens = tokenizer(
            dom_text,
            max_length=DOM_MAX_LEN,
            truncation=True,
            return_tensors=None,
        )
        truncated_text = tokenizer.decode(tokens['input_ids'], skip_special_tokens=True)

        t0 = time.time()
        payload = json.dumps({"inputs": truncated_text})
        resp = sm_runtime.invoke_endpoint(
            EndpointName=DOM_ENDPOINT_NAME,
            ContentType='application/json',
            Accept='application/json',
            Body=payload,
        )
        raw = json.loads(resp['Body'].read().decode('utf-8'))
        latency = round((time.time() - t0) * 1000, 1)
        logger.info(f"[DOM Model] raw={raw} latency={latency}ms")

        if isinstance(raw, list):
            raw = raw[0]

        label = str(raw.get('label', 'LABEL_0')).strip()
        score = float(raw.get('score', 0.5))
        prob_phishing = score if label in ('LABEL_1', 'M', '1', 1) else 1.0 - score
        return round(prob_phishing, 4), raw

    except Exception as e:
        logger.error(f"[DOM Model] Error: {e}")
        return None, {"error": str(e)}

In [ ]:
# ── Stage 2: Claude VLM analysis (conditional) ──────────────────────────────

def run_vlm_analysis(screenshot_b64: Optional[str], url: str, html: str) -> tuple[float, str]:
    """
    Conditionally invoke a VLM (Claude) on a screenshot or rendered HTML snippet.
    Uses the Amazon Bedrock Converse API.
    Returns (prob_phishing_0_to_1, reasoning_text).

    If screenshot_b64 is None, falls back to a text-only prompt.
    """
    try:
        t0 = time.time()

        try:
            soup = BeautifulSoup(html, 'lxml')
            for tag in soup(['script', 'style', 'noscript']):
                tag.decompose()
            visible_text = soup.get_text(separator=' ', strip=True)[:800]
        except Exception:
            visible_text = html[:800]

        analysis_prompt = (
            f"""You are a phishing detection expert. Analyze this webpage.

            URL: {url}
            Visible page text: {visible_text}


            Assess whether this page is a phishing attempt. Consider:

            - Brand impersonation (logos, colors, layout mimicking legitimate sites)

            - Urgency/fear tactics (account suspended, verify now, tax refund)

            - Credential harvesting forms (login, password, SSN, credit card fields)

            - Suspicious external links or redirects

            - Mismatch between claimed brand and URL domain


            Respond in JSON only:

            f'{{"phishing_probability": <0.0-1.0>, "confidence": "high|medium|low", '
            f'"reasoning": "<1-2 sentence explanation>", '
            f'"indicators": ["<indicator1>", "<indicator2>"]}}'"""
        )
        
        if screenshot_b64:
            content = [
                {
                    "image": {
                        "format": "png",
                        "source": {
                            "bytes": base64.b64decode(screenshot_b64)
                        }
                    }
                },
                {
                    "text": analysis_prompt
                }
            ]
        else:
            content = [
                {
                    "text": analysis_prompt
                }
            ]

        response = bedrock_rt.converse(
            modelId=VLM_MODEL_ID,
            system=[
                {
                    "text": (
                        "You are a cybersecurity expert specializing in phishing detection. "
                        "Always respond with valid JSON only — no markdown, no explanation outside the JSON object."
                    )
                }
            ],
            messages=[
                {
                    "role": "user",
                    "content": content
                }
            ],
            inferenceConfig={
                "maxTokens": 512,
                "temperature": 0.0,   
            }
        )

        latency = round((time.time() - t0) * 1000, 1)

        response_text = response["output"]["message"]["content"][0]["text"]

        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            vlm_result = json.loads(json_match.group())
            prob       = float(vlm_result.get('phishing_probability', 0.5))
            reasoning  = vlm_result.get('reasoning', '')
            indicators = vlm_result.get('indicators', [])
            full_reasoning = f"{reasoning} Indicators: {', '.join(indicators)}"
            logger.info(f"[VLM] prob={prob:.4f} latency={latency}ms reasoning={reasoning}")
            return round(prob, 4), full_reasoning
        else:
            logger.warning(f"[VLM] Could not parse JSON from response: {response_text[:200]}")
            return 0.5, response_text[:200]

    except Exception as e:
        logger.error(f"[VLM] Error: {e}")
        return 0.5, f"VLM error: {str(e)}"


In [ ]:
# ── Decision engine: weighted score aggregation ─────────────────────────────

def compute_combined_score(
    url_score:  Optional[float],
    dom_score:  Optional[float],
    vlm_score:  Optional[float] = None,
) -> float:
    """
    Weighted combination of available model scores.
    Weights: URL=0.35, DOM=0.45, VLM=0.20 (when triggered)
    If a model failed (None), redistribute its weight proportionally.
    """
    scores  = []
    weights = []

    if url_score is not None:
        scores.append(url_score)
        weights.append(0.35)
    if dom_score is not None:
        scores.append(dom_score)
        weights.append(0.45)
    if vlm_score is not None:
        scores.append(vlm_score)
        weights.append(0.20)

    if not scores:
        return 0.5  # no signal → uncertain

    total_weight = sum(weights)
    combined = sum(s * w for s, w in zip(scores, weights)) / total_weight
    return round(combined, 4)


def make_decision(combined_score: float) -> Decision:
    if combined_score >= HIGH_CONFIDENCE_THRESHOLD:
        return Decision.AUTO_BLOCK
    elif combined_score <= LOW_CONFIDENCE_THRESHOLD:
        return Decision.ALLOW
    else:
        return Decision.HUMAN_REVIEW



In [ ]:
# ── Pipeline orchestrator ───────────────────────────────────────────────────

def run_phishing_pipeline(
    url:             str,
    html:            str,
    screenshot_b64:  Optional[str] = None,
    enable_vlm:      bool = True,
) -> PipelineResult:
    """
    Full phishing detection pipeline:
      Stage 1a: LightGBM URL classifier
      Stage 1b: DistilBERT DOM classifier
      Stage 2 (conditional): VLM screenshot/text analysis
      Decision Engine: weighted score → AUTO_BLOCK / HUMAN_REVIEW / ALLOW

    Args:
        url:            The URL of the page being analyzed
        html:           Raw HTML content of the page
        screenshot_b64: Base64-encoded PNG screenshot (optional; VLM falls back to text if None)
        enable_vlm:     Set False to disable VLM stage entirely (e.g., for batch eval)
    """
    t_pipeline_start = time.time()
    result = PipelineResult(url=url, decision=Decision.ALLOW, combined_score=0.0)

    t0 = time.time()
    url_score, url_raw = run_url_model(url)
    result.url_score    = url_score
    result.url_model_raw = url_raw
    result.latency_ms['url_model'] = round((time.time() - t0) * 1000, 1)

    if url_score is None:
        result.errors.append('url_model_failed')

    t0 = time.time()
    dom_score, dom_raw = run_dom_model(html, url)
    result.dom_score    = dom_score
    result.dom_model_raw = dom_raw
    result.latency_ms['dom_model'] = round((time.time() - t0) * 1000, 1)

    if dom_score is None:
        result.errors.append('dom_model_failed')

    prelim_score = compute_combined_score(url_score, dom_score)

    # ── Stage 2 (Conditional): VLM ───────────────────────────────────────────
    # Trigger VLM if:
    #   1. VLM is enabled
    #   2. Score is in the medium-confidence band (not already auto-block or allow)
    #   3. At least one layer flagged suspicious (score > VLM_TRIGGER_THRESHOLD)
    should_trigger_vlm = (
        enable_vlm
        and LOW_CONFIDENCE_THRESHOLD < prelim_score < HIGH_CONFIDENCE_THRESHOLD
        and prelim_score > VLM_TRIGGER_THRESHOLD
    )

    if should_trigger_vlm:
        logger.info(f"[Pipeline] VLM triggered (prelim_score={prelim_score:.4f})")
        t0 = time.time()
        vlm_score, vlm_reasoning = run_vlm_analysis(screenshot_b64, url, html)
        result.vlm_score     = vlm_score
        result.vlm_triggered = True
        result.vlm_reasoning = vlm_reasoning
        result.latency_ms['vlm'] = round((time.time() - t0) * 1000, 1)
    else:
        logger.info(f"[Pipeline] VLM skipped (prelim_score={prelim_score:.4f}, enable_vlm={enable_vlm})")

    result.combined_score = compute_combined_score(url_score, dom_score, result.vlm_score)
    result.decision       = make_decision(result.combined_score)
    result.latency_ms['total'] = round((time.time() - t_pipeline_start) * 1000, 1)

    return result


def print_result(r: PipelineResult):
    decision_emoji = {'AUTO_BLOCK': '🔴', 'HUMAN_REVIEW': '🟡', 'ALLOW': '🟢'}
    print(f"{'='*60}")
    print(f"  URL      : {r.url[:70]}")
    print(f"  Decision : {decision_emoji.get(r.decision, '⚪')} {r.decision}")
    print(f"  Combined : {r.combined_score:.4f}")
    print(f"  URL Score: {r.url_score if r.url_score is not None else 'N/A (failed)'}")
    print(f"  DOM Score: {r.dom_score if r.dom_score is not None else 'N/A (failed)'}")
    if r.vlm_triggered:
        print(f"  VLM Score: {r.vlm_score:.4f}  ← triggered")
        print(f"  VLM Reason: {r.vlm_reasoning}")
    else:
        print(f"  VLM Score: not triggered")
    print(f"  Latency  : URL={r.latency_ms.get('url_model','?')}ms  "
          f"DOM={r.latency_ms.get('dom_model','?')}ms  "
          f"VLM={r.latency_ms.get('vlm','not run')}  "
          f"Total={r.latency_ms.get('total','?')}ms")
    if r.errors:
        print(f"  Errors   : {r.errors}")
    print(f"{'='*60}")


In [ ]:
# ── Test harness ─────────────────────────────────────────────────────────────

if __name__ == '__main__':

    phishing_html = """<!DOCTYPE html>
<html><head><title>Chase Bank - Urgent Account Verification Required</title>
<meta name="description" content="Chase Bank secure account verification portal">
</head><body>
<div class="alert-banner">SECURITY ALERT: Suspicious login activity detected.</div>
<form method="POST" action="https://collect-creds.invalid/chase/submit">
  <input type="email" name="email" placeholder="Email Address">
  <input type="password" name="password" placeholder="Password">
  <input type="hidden" name="redirect" value="https://chase.com">
  <button type="submit">Verify Account</button>
</form>
<script src="https://cdn-assets.secure-banking-resources.invalid/chase/track.js"></script>
</body></html>"""

    print("[TEST 1] Known phishing page — Chase Bank fake login")
    r1 = run_phishing_pipeline(
        url='https://chase-bank-phish.invalid/verify',
        html=phishing_html,
        screenshot_b64=None,   # no screenshot for this test
        enable_vlm=True,
    )
    print_result(r1)

    benign_html = """<!DOCTYPE html>
<html><head><title>Welcome to Our Blog</title>
<meta name="description" content="A blog about technology and design">
</head><body>
<h1>Latest Articles</h1>
<p>Read our latest posts about web development, design trends, and more.</p>
<a href="/articles/intro-to-css">Introduction to CSS</a>
<a href="/articles/js-tips">JavaScript Tips</a>
</body></html>"""

    print("[TEST 2] Benign page — tech blog")
    r2 = run_phishing_pipeline(
        url='https://techblog.example.com/home',
        html=benign_html,
        screenshot_b64=None,
        enable_vlm=True,
    )
    print_result(r2)

    lure_html = """<!DOCTYPE html>
<html><head><title>IRS - Important Tax Refund Notice | Internal Revenue Service</title>
<meta name="description" content="Your federal tax refund is pending verification">
</head><body>
<div class="alert">IMPORTANT: Your $2,847 tax refund requires immediate verification.</div>
<p>Our records indicate you have an unclaimed federal tax refund. 
   To receive your refund, you must verify your identity within 48 hours.</p>
<a href="https://irs-refund-verify.invalid/claim?token=abc123">
  Click Here to Claim Your Refund Now
</a>
<p>Failure to verify within 48 hours will result in forfeiture of your refund.</p>
</body></html>"""

    print("[TEST 3] Lure-style phishing — IRS tax refund (no credential form)")
    r3 = run_phishing_pipeline(
        url='https://irs-refund-verify.invalid/notice',
        html=lure_html,
        screenshot_b64=None,
        enable_vlm=True,   # VLM should help catch this lure-style page
    )
    print_result(r3)

    print("[TEST 4] Same lure page — VLM disabled (batch eval mode)")
    r4 = run_phishing_pipeline(
        url='https://irs-refund-verify.invalid/notice',
        html=lure_html,
        screenshot_b64=None,
        enable_vlm=False,  # skip VLM for speed
    )
    print_result(r4)

    print("── Pipeline Summary ─────────────────────────────────────")
    for i, r in enumerate([r1, r2, r3, r4], 1):
        print(f"  Test {i}: {r.decision:15s} | score={r.combined_score:.4f} | "
              f"url={r.url_score} dom={r.dom_score} vlm={r.vlm_score} | "
              f"total={r.latency_ms.get('total','?')}ms")


---
## Cleanup

Amazon SageMaker AI real-time endpoints incur cost **while running**, even with zero traffic.
Run the cell below when you are done to avoid ongoing charges.
You can always redeploy by re-running the training notebooks.

In [ ]:
import boto3

sm_client = boto3.client('sagemaker', region_name=AWS_REGION)

for endpoint_name in [URL_ENDPOINT_NAME, DOM_ENDPOINT_NAME]:
    if endpoint_name.startswith('<'):
        print(f'Skipping placeholder: {endpoint_name}')
        continue
    try:
        sm_client.delete_endpoint(EndpointName=endpoint_name)
        print(f'Deleted endpoint: {endpoint_name}')
    except sm_client.exceptions.ClientError as e:
        print(f'Could not delete {endpoint_name}: {e}')


---
## Cost Estimate

The table below shows approximate AWS costs for running this pipeline.
All prices are **us-east-1** on-demand rates (April 2026) — verify current rates at
[aws.amazon.com/sagemaker/pricing](https://aws.amazon.com/sagemaker/pricing/) and
[aws.amazon.com/bedrock/pricing](https://aws.amazon.com/bedrock/pricing/).

| Component | Instance | $/hr | Notes |
|---|---|---|---|
| URL endpoint (LightGBM) | ml.m5.large | $0.115 | Always-on real-time endpoint |
| DOM endpoint (DistilBERT) | ml.m5.xlarge | $0.230 | Always-on real-time endpoint |
| Amazon Bedrock Claude (VLM) | — | $0.003 / 1k input tokens | ~500 tokens/call → ~$0.0015/call |

The code cell below calculates monthly costs for several traffic scenarios.

In [ ]:
PRICE_URL_ENDPOINT_PER_HR  = 0.115   # ml.m5.large
PRICE_DOM_ENDPOINT_PER_HR  = 0.230   # ml.m5.xlarge
PRICE_BEDROCK_PER_1K_INPUT = 0.003   # Claude Sonnet — input tokens
PRICE_BEDROCK_PER_1K_OUTPUT= 0.015   # Claude Sonnet — output tokens

HOURS_PER_MONTH = 730
ENDPOINT_MONTHLY = (PRICE_URL_ENDPOINT_PER_HR + PRICE_DOM_ENDPOINT_PER_HR) * HOURS_PER_MONTH

VLM_INPUT_TOKENS  = 500   # URL + visible page text prompt
VLM_OUTPUT_TOKENS = 120   # JSON response with reasoning
PRICE_PER_VLM_CALL = (
    (VLM_INPUT_TOKENS  / 1000) * PRICE_BEDROCK_PER_1K_INPUT +
    (VLM_OUTPUT_TOKENS / 1000) * PRICE_BEDROCK_PER_1K_OUTPUT
)

print('=' * 62)
print('  MONTHLY ENDPOINT COST (always-on)')
print('=' * 62)
print(f'  URL endpoint  (ml.m5.large)   ${PRICE_URL_ENDPOINT_PER_HR:.3f}/hr  x {HOURS_PER_MONTH}h = ${PRICE_URL_ENDPOINT_PER_HR * HOURS_PER_MONTH:7.2f}/mo')
print(f'  DOM endpoint  (ml.m5.xlarge)  ${PRICE_DOM_ENDPOINT_PER_HR:.3f}/hr  x {HOURS_PER_MONTH}h = ${PRICE_DOM_ENDPOINT_PER_HR * HOURS_PER_MONTH:7.2f}/mo')
print(f'  Endpoint total                                     ${ENDPOINT_MONTHLY:7.2f}/mo')
print()
print('=' * 62)
print('  PER-INVOCATION COST')
print('=' * 62)
print(f'  URL model     — SageMaker endpoint   ~$0.00 (covered by hourly rate)')
print(f'  DOM model     — SageMaker endpoint   ~$0.00 (covered by hourly rate)')
print(f'  VLM (Claude)  — Amazon Bedrock per call      ~${PRICE_PER_VLM_CALL:.4f}')
print(f'    ({VLM_INPUT_TOKENS} input + {VLM_OUTPUT_TOKENS} output tokens estimated)')
print()

print('=' * 62)
print('  TOTAL MONTHLY COST BY TRAFFIC VOLUME')
print('=' * 62)
print(f'  {"URLs/mo":<15} {"VLM rate":<12} {"VLM cost":<14} {"Total/mo"}')
print(f'  {"-"*15} {"-"*12} {"-"*14} {"-"*10}')

scenarios = [
    (10_000,    0.10),
    (50_000,    0.10),
    (100_000,   0.10),
    (500_000,   0.05),
    (1_000_000, 0.05),
]

for urls_per_month, vlm_rate in scenarios:
    vlm_calls   = urls_per_month * vlm_rate
    vlm_cost    = vlm_calls * PRICE_PER_VLM_CALL
    total_cost  = ENDPOINT_MONTHLY + vlm_cost
    cost_per_url = total_cost / urls_per_month * 1000  # per 1000 URLs
    print(f'  {urls_per_month:<15,} {vlm_rate*100:<11.0f}% '
          f'${vlm_cost:<13.2f} ${total_cost:.2f}/mo  '
          f'(~${cost_per_url:.3f} per 1k URLs)')

print()
print('  Notes:')
print('  - SageMaker AI endpoint cost is fixed (hourly) regardless of traffic volume.')
print('  - VLM is only triggered for medium-confidence URLs (score 0.25–0.85).')
print('  - Use SageMaker AI Serverless Inference to eliminate idle endpoint cost for')
print('    low-volume or bursty workloads (cold start ~1–2s tradeoff).')
print('  - Prices are approximate; verify current rates at aws.amazon.com/pricing')
